# SceneFlow Dataset Downloader & Reformatter for Google Colab
This notebook automates downloading the SceneFlow datasets (FlyingThings3D and Driving) using high-speed direct HTTP connections (`aria2c`) and extracting them directly to your mounted Google Drive in the format expected by the `bridgedepth` data loader.

## Core Optimizations:
1. **Direct HTTP Downloads**: Replaced slow and blocked BitTorrent/P2P downloads with high-speed direct HTTP links from the University of Freiburg.
2. **Parallel Segments**: Configured `aria2c` with 16 connections and split segment downloads to maximize download speed (often 50-100+ MB/s).
3. **Zero Pre-allocation**: Disabled file allocation (`--file-allocation=none`) to avoid writing gigabytes of zero blocks and wasting disk I/O.
4. **FUSE Cache Purging**: Google Drive FUSE mount caches all written files locally on the VM under `/root/.config/Google/DriveFS`. To prevent this cache from filling the Colab VM disk, this script **flushes and unmounts the Drive after each archive extraction**, completely purging the local cache, and then automatically remounts it for the next job.
5. **Casing Guard**: Automatically renames directories to ensure `driving` is lowercase and `FlyingThings3D` is capitalized, preventing dataloader glob mismatch errors.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Install aria2 downloader (very fast, supports parallel HTTP downloads)
!apt-get update && apt-get install -y aria2

In [ ]:
import os
import urllib.parse
import tarfile
from pathlib import Path

# ── Target Directory inside Google Drive ──────────────────────────────────
# The bridgedepth data loader expects: 
#  - FlyingThings3D images inside: sceneflow_root/FlyingThings3D/frames_cleanpass/ or frames_finalpass/
#  - driving images inside: sceneflow_root/driving/frames_cleanpass/ or frames_finalpass/
GDRIVE_SCENEFLOW_ROOT = "/content/drive/MyDrive/sceneflow"

# Temp directory on local Colab disk to store compressed downloads
LOCAL_TEMP_DIR = "/content/temp_downloads"

# List of download jobs. Set enable=False if you want to skip any specific parts.
# Note: Direct HTTP URLs are derived by removing '.torrent' from the official torrent links.
DOWNLOAD_JOBS = [
    {
        "name": "FlyingThings3D Disparity",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/FlyingThings3D/derived_data/flyingthings3d__disparity.tar.bz2",
        "filename": "flyingthings3d__disparity.tar.bz2",
        "expected_dir": "FlyingThings3D",
        "enabled": True
    },
    {
        "name": "FlyingThings3D RGB Cleanpass Images",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/FlyingThings3D/raw_data/flyingthings3d__frames_cleanpass.tar",
        "filename": "flyingthings3d__frames_cleanpass.tar",
        "expected_dir": "FlyingThings3D",
        "enabled": True
    },
    {
        "name": "FlyingThings3D RGB Finalpass Images",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/FlyingThings3D/raw_data/flyingthings3d__frames_finalpass.tar",
        "filename": "flyingthings3d__frames_finalpass.tar",
        "expected_dir": "FlyingThings3D",
        "enabled": True
    },
    {
        "name": "FlyingThings3D Camera Data",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/CameraData_august16/data/FlyingThings3D/raw_data/flyingthings3d__camera_data.tar",
        "filename": "flyingthings3d__camera_data.tar",
        "expected_dir": "FlyingThings3D",
        "enabled": True
    },
    {
        "name": "Driving RGB Cleanpass Images",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/Driving/raw_data/driving__frames_cleanpass.tar",
        "filename": "driving__frames_cleanpass.tar",
        "expected_dir": "driving",
        "enabled": True
    },
    {
        "name": "Driving RGB Finalpass Images",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/Driving/raw_data/driving__frames_finalpass.tar",
        "filename": "driving__frames_finalpass.tar",
        "expected_dir": "driving",
        "enabled": True
    },
    {
        "name": "Driving Disparity",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/Driving/derived_data/driving__disparity.tar.bz2",
        "filename": "driving__disparity.tar.bz2",
        "expected_dir": "driving",
        "enabled": True
    },
    {
        "name": "Driving Camera Data",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/CameraData_august16/data/Driving/raw_data/driving__camera_data.tar",
        "filename": "driving__camera_data.tar",
        "expected_dir": "driving",
        "enabled": True
    }
]

In [ ]:
def download_file(url: str, filename: str, temp_dir: str):
    os.makedirs(temp_dir, exist_ok=True)
    local_path = os.path.join(temp_dir, filename)
    
    # Clean up existing files if they exist
    if os.path.exists(local_path):
        print(f"[downloader] File already exists locally, removing: {local_path}")
        os.remove(local_path)
        
    print(f"[downloader] Initiating direct HTTP download for: {filename}")
    print(f"[downloader] URL: {url}")
    # aria2c command optimized for high-speed direct downloading:
    # -x 16: use up to 16 connections per server
    # -s 16: split the file into 16 parts to parallelize download segments
    # -k 1M: split chunk size threshold of 1MB
    # --file-allocation=none: disables file allocation to avoid writing gigabytes of zeros first
    # -c: continue partially downloaded file
    cmd = f"aria2c -x 16 -s 16 -k 1M --file-allocation=none -c -d {temp_dir} -o {filename} '{url}'"
    status = os.system(cmd)
    if status != 0:
        raise RuntimeError(f"Direct HTTP download failed for {filename}")
            
    return local_path


def extract_and_format_archive(archive_path: str, dest_root: str, expected_dir: str):
    print(f"[extractor] Inspecting archive: {archive_path}")
    
    # Check top-level folder inside tar archive
    with tarfile.open(archive_path, 'r') as tar:
        first_member = tar.next()
        if first_member:
            top_dir = first_member.name.split('/')[0]
        else:
            top_dir = ""
            
    # Decide extraction folder layout to match dataloader structure
    if top_dir.lower() == expected_dir.lower():
        extract_path = dest_root
    else:
        extract_path = os.path.join(dest_root, expected_dir)
        
    os.makedirs(extract_path, exist_ok=True)
    print(f"[extractor] Extracting directory layout to Google Drive destination: {extract_path}")
    
    # Use highly optimized Unix tar CLI command
    if archive_path.endswith('.bz2'):
        cmd = f"tar -xjf {archive_path} -C {extract_path}"
    else:
        cmd = f"tar -xf {archive_path} -C {extract_path}"
        
    print(f"[extractor] Executing extraction command: {cmd}")
    status = os.system(cmd)
    if status != 0:
        raise RuntimeError(f"Extraction failed for {archive_path}")
    print(f"[extractor] Successfully extracted!")

    # Casing correction: ensure driving/FlyingThings3D folder names match exactly
    # Rename Driving -> driving if necessary
    driving_cap = os.path.join(dest_root, "Driving")
    driving_low = os.path.join(dest_root, "driving")
    if os.path.exists(driving_cap) and not os.path.exists(driving_low):
        print(f"[casing] Renaming '{driving_cap}' -> '{driving_low}' to match dataloader expectation.")
        os.rename(driving_cap, driving_low)
        
    # Rename flyingthings3d -> FlyingThings3D if necessary
    things_low = os.path.join(dest_root, "flyingthings3d")
    things_cap = os.path.join(dest_root, "FlyingThings3D")
    if os.path.exists(things_low) and not os.path.exists(things_cap):
        print(f"[casing] Renaming '{things_low}' -> '{things_cap}' to match dataloader expectation.")
        os.rename(things_low, things_cap)

In [ ]:
import time
import shutil
from google.colab import drive

if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(GDRIVE_SCENEFLOW_ROOT, exist_ok=True)

for idx, job in enumerate(DOWNLOAD_JOBS):
    if not job["enabled"]:
        print(f"\n>>> Skipping job {idx+1}/{len(DOWNLOAD_JOBS)}: {job['name']}")
        continue
        
    print("\n" + "=" * 80)
    print(f"  JOB {idx+1}/{len(DOWNLOAD_JOBS)}: {job['name']}")
    print("=" * 80)
    
    # 0. Check and mount drive if unmounted
    if not os.path.exists("/content/drive/MyDrive"):
        print("[gdrive] Mount not active. Remounting Google Drive...")
        drive.mount('/content/drive')
        os.makedirs(GDRIVE_SCENEFLOW_ROOT, exist_ok=True)
        
    try:
        # 1. Download file to local VM SSD
        local_archive = download_file(
            url=job["url"],
            filename=job["filename"],
            temp_dir=LOCAL_TEMP_DIR
        )
        
        # 2. Extract directly to Google Drive
        extract_and_format_archive(
            archive_path=local_archive,
            dest_root=GDRIVE_SCENEFLOW_ROOT,
            expected_dir=job["expected_dir"]
        )
        
        # 3. Clean up local files
        print(f"[cleanup] Deleting local archive to save disk space: {local_archive}")
        if os.path.exists(local_archive):
            os.remove(local_archive)
        if os.path.exists(LOCAL_TEMP_DIR):
            shutil.rmtree(LOCAL_TEMP_DIR, ignore_errors=True)
            
        # 4. Flush Google Drive FUSE cache and unmount to release local disk space
        print("[cleanup] Flushing Google Drive cache and unmounting to release cached disk space...")
        drive.flush_and_unmount()
        
        # Clean up any leftover DriveFS metadata cache folder
        drivefs_cache = "/root/.config/Google/DriveFS"
        if os.path.exists(drivefs_cache):
            shutil.rmtree(drivefs_cache, ignore_errors=True)
            print("[cleanup] DriveFS local write cache cleared.")
            
        print("[cleanup] Sleep for 5 seconds to ensure system registers unmount...")
        time.sleep(5)
            
    except Exception as e:
        print(f"[ERROR] Job {job['name']} failed: {e}")
        print("Stopping pipeline. Please resolve and resume.")
        break

# Final remount so the user can inspect files in file explorer
if not os.path.exists("/content/drive/MyDrive"):
    print("\n[gdrive] Performing final mount of Google Drive...")
    drive.mount('/content/drive')
        
print("\n" + "=" * 80)
print("  PIPELINE PROCESSING COMPLETED")
print("=" * 80)
